In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import os

def run_cmd(cmd, env=None):
    print(' '.join(cmd))
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'Failed with code {proc.returncode}')
    return proc

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
PYTHON = '/home/bzhang512/miniconda3/envs/reggs/bin/python'
SCRIPT = Path('/home/bzhang512/CV_Project/part3_BRPO/scripts/prepare_stage1_difix_dataset_s3po.py')
SCENE_NAME = 'Re10k-1'
RUN_KEY = 're10k1__s3po__midpoint__v1'
DATASET_ROOT = Path('/home/bzhang512/CV_Project/dataset')
STORAGE_ROOT = Path('/home/bzhang512/my_storage_500G/CV_Project')
EXTERNAL_EVAL_RUN = '2026-04-04-02-11-09'
EXTERNAL_EVAL_DIR = STORAGE_ROOT / 'output/part2_s3po/00_external_eval/re10k1/infer_re10k1_full' / EXTERNAL_EVAL_RUN
SPARSE_MANIFEST = DATASET_ROOT / SCENE_NAME / 'part2_s3po/sparse/split_manifest.json'
TEST_MANIFEST = DATASET_ROOT / SCENE_NAME / 'part2_s3po/test/split_manifest.json'
SPARSE_RGB_DIR = DATASET_ROOT / SCENE_NAME / 'part2_s3po/sparse/rgb'
RENDER_RGB_DIR = EXTERNAL_EVAL_DIR / 'render_rgb'
RENDER_DEPTH_DIR = EXTERNAL_EVAL_DIR / 'render_depth_npy'
TRJ_JSON = EXTERNAL_EVAL_DIR / 'plot/trj_external_infer.json'
PLACEMENT = 'midpoint'
PROMPT = 'remove degradation'
DIFIX_MODEL_NAME = 'nvidia/difix_ref'
HEIGHT = 512
WIDTH = 512
TIMESTEP = 199
print('SCENE_NAME:', SCENE_NAME)
print('EXTERNAL_EVAL_DIR:', EXTERNAL_EVAL_DIR)

In [ ]:
# Check
for p in [SPARSE_MANIFEST, TEST_MANIFEST, SPARSE_RGB_DIR, RENDER_RGB_DIR, RENDER_DEPTH_DIR, TRJ_JSON]:
    assert p.exists(), f'Missing {p}'
print('All inputs OK')
sparse = read_json(SPARSE_MANIFEST)
test = read_json(TEST_MANIFEST)
print(f'Sparse: {len(sparse["selected_indices"])} frames')
print(f'Test: {len(test["selected_indices"])} frames')

In [ ]:
# Run all stages
env = dict(os.environ)
env['HF_ENDPOINT'] = 'https://hf-mirror.com'
cmd = [PYTHON, str(SCRIPT), '--stage', 'all', '--scene-name', SCENE_NAME, '--run-key', RUN_KEY,
       '--dataset-root', str(DATASET_ROOT), '--sparse-manifest', str(SPARSE_MANIFEST),
       '--test-manifest', str(TEST_MANIFEST), '--sparse-rgb-dir', str(SPARSE_RGB_DIR),
       '--render-rgb-dir', str(RENDER_RGB_DIR), '--render-depth-dir', str(RENDER_DEPTH_DIR),
       '--trj-json', str(TRJ_JSON), '--placement', PLACEMENT, '--difix-model-name', DIFIX_MODEL_NAME,
       '--prompt', PROMPT, '--height', str(HEIGHT), '--width', str(WIDTH), '--timestep', str(TIMESTEP)]
run_cmd(cmd, env)

In [ ]:
# Check results
RUN_ROOT = DATASET_ROOT / SCENE_NAME / 'part3_stage1' / RUN_KEY
print('left_fixed:', len(list((RUN_ROOT/'difix/left_fixed').glob('*.png'))))
print('right_fixed:', len(list((RUN_ROOT/'difix/right_fixed').glob('*.png'))))
print('aug_left:', len(list((RUN_ROOT/'augmented_train_left/rgb').glob('*.png'))))
cache = read_json(RUN_ROOT / 'pseudo_cache/manifest.json')
print('pseudo samples:', cache['num_samples'])
print('sample_ids:', cache['sample_ids'])